In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [23]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects. Enclose it in <json> tags and when cleaned upon clean up and dumping into a json file it should be a array of prompots do format accordingly
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, stop_sequences=["</json>"])
    json_text = text.replace("<json>", "").strip()
    return json_text

dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(json.loads(dataset), f, indent=2)

In [37]:
with open('dataset.json', 'r') as f:
    dataset = json.load(f)

In [38]:
type(dataset)

list

In [39]:
dataset

[{'task': "Write a Python function that parses an AWS S3 bucket name from an S3 URI in the format 's3://bucket-name/key/path' and returns just the bucket name."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific DynamoDB table named 'users-table'."},
 {'task': 'Write a regular expression that matches and extracts AWS ARN (Amazon Resource Name) format strings, specifically for EC2 instances in any region.'}]

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}

 Enclose the response in <json> tags
    """

    messages = []
    add_user_message(messages, eval_prompt)
    text = chat(messages, stop_sequences=["</json>"])
    json_text = text.replace("<json>", "").strip()
    print(json.loads(json_text))



In [40]:
grade_by_model(dataset[0], "def list_s3_buckets():\n    import boto3\n    s3 = boto3.client('s3')\n    response = s3.list_buckets()\n    return [bucket['Name'] for bucket in response['Buckets']]")


{'strengths': ['Uses the correct AWS SDK (boto3) for AWS operations',
  'Proper error handling structure with boto3 client initialization'],
 'weaknesses': ['Completely ignores the task requirement - should parse S3 URI strings, not list buckets from AWS',
  'Does not accept any input parameter (should accept an S3 URI string)',
  "Does not extract bucket name from URI format 's3://bucket-name/key/path'",
  'Makes unnecessary AWS API calls instead of performing simple string parsing',
  'Requires AWS credentials and network access when simple string parsing would suffice'],
 'reasoning': "The solution is fundamentally misaligned with the task requirements. The task asks for a function that parses an S3 URI string to extract the bucket name (a simple string manipulation operation), but the solution implements a completely different function that lists all S3 buckets from AWS. While the code itself is syntactically correct, it solves an entirely different problem. A correct solution woul

In [ ]:
type(dataset[0])

In [27]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [33]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [29]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [36]:
results = run_eval(dataset)

JSONDecodeError: Invalid \escape: line 9 column 77 (char 928)

In [31]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses an S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parses an AWS S3 bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): An S3 URI in the format 's3://bucket-name/key/path'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n    \n    Example:\n        >>> parse_s3_bucket_name('s3://my-bucket/path/to/file.txt')\n        'my-bucket'\n    \"\"\"\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI: {s3_uri}. Must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split by '/' and get the first part (bucket name)\n    bucket_name = uri_without_prefix.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(f\"Invalid S3 URI: {s3_uri}. No bucket name foun